# Amazon ML Challenge 2026 — Official End-to-End GPU Training & Submission Pipeline

**Architecture Highlights:**
- **Dense FAISS GPU Semantic Retrieval**: `all-MiniLM-L6-v2` dense vectors indexed with GPU `IndexFlatIP`
- **Multi-Core Preprocessing**: Parallel transliteration & regex-based postal code/address normalization
- **12-Dimensional Pairwise Feature Engineering**: Levenshtein, Jaccard, Jaro-Winkler, PIN, Number agreement, City, State
- **Cost-Sensitive LightGBM Classifier**: Trained with balanced positive weighting for Macro $F_{0.5}$
- **Optimal Threshold Sweep**: Sweeps $[0.20, 0.85]$ to maximize the official competition metric
- **Full Test Set Inference**: Generates official `matching_results.tsv` and `candidate_pairs.tsv`
- **Automated Validation**: Verifies against `validate_submission.py` format specifications


In [ ]:
# 1. Install GPU and ML dependencies
!pip install -q polars lightgbm sentence-transformers faiss-gpu jellyfish kaggle tqdm
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 2. Clone the latest Model_Mania repository
!rm -rf Model_Mania
!git clone https://github.com/Naman-y/Model_Mania.git
%cd Model_Mania
!git log -1 --oneline


In [ ]:
# 3. Download the full Amazon ML Challenge 2026 Dataset from Kaggle
import os
os.environ['KAGGLE_USERNAME'] = 'golok7'
os.environ['KAGGLE_KEY'] = '6fe02ca9027ff9560167aa5cdf9667f8'

os.makedirs('student_resource/dataset', exist_ok=True)
!kaggle datasets download -d satwiksps/amazon-ml-challenge-2026 -p /tmp/dataset_zip --unzip

# Link or copy dataset to standard location
if os.path.exists('/tmp/dataset_zip/dataset'):
    !cp -rn /tmp/dataset_zip/dataset/* student_resource/dataset/
elif os.path.exists('/tmp/dataset_zip/train'):
    !cp -rn /tmp/dataset_zip/* student_resource/dataset/

!ls -lh student_resource/dataset/train


In [ ]:
# 4. Launch Full GPU Training with Dense FAISS, Threshold Optimization & Test Submission
!python kaggle_deploy/solution_bundled.py


In [ ]:
# 5. Validate the Generated Submissions & Create Downloadable Zip
!python student_resource/utils/validate_submission.py --matching output/matching_results_train.tsv --candidate output/candidate_pairs_train.tsv --test-dir student_resource/dataset/train

!cd output && zip -9 submission.zip matching_results_train.tsv candidate_pairs_train.tsv entity_match_model.joblib
!ls -lh output/submission.zip
print('\nSUCCESS: Pipeline completed! Download output/submission.zip from the Colab file browser.')
